# Experiments with Stable Diffusion XL (SDXL) using the `diffusers` library

Adapted by Antonio Esteves, UMinho

* * * 

[Stable Diffusion XL](https://huggingface.co/papers/2307.01952) (SDXL) is a powerful text-to-image generation model that improves the previous Stable Diffusion models in four key ways:

1. the U-Net is 3x larger;
2. it adds a second text encoder (OpenCLIP ViT-bigG/14) that will complement the original text encoder;
3. it introduces size- and crop-conditioning to allow generate images of diverse aspect-ratios and to solve the cropping issue related to image cropping during training of Stabele Diffusion;
4. adopts a two-stage model pipeline, where the *base* model (which can also be run as a standalone model) generates a latent that is used as an input to the *refiner* model, which adds additional high-quality details.

This notebook shows how to use SDXL for text-to-image, image-to-image, and inpainting taks.

Before we begin, make sure that the following libraries are installed:

In [ ]:
# uncomment to install the necessary libraries in Colab
!pip install -q diffusers transformers accelerate invisible-watermark>=0.2.0
!pip install --upgrade transformers


It is recommended installing the [invisible-watermark](https://pypi.org/project/invisible-watermark/) library to help identify images that are generated. If the invisible-watermark library is installed, it is used by default. To disable the watermarker:

```py
pipeline = StableDiffusionXLPipeline.from_pretrained(..., add_watermarker=False)
```

## Load the model checkpoints

Model weights may be stored in separate subfolders on the Hugging Face hub or locally, in which case, we should use the [from_pretrained()](https://huggingface.co/docs/diffusers/main/en/api/pipelines/overview#diffusers.DiffusionPipeline.from_pretrained) method.

In [ ]:
from   diffusers import StableDiffusionXLPipeline, StableDiffusionXLImg2ImgPipeline
import torch

pipeline = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype     = torch.float16,
    variant         = "fp16",
    use_safetensors = True,
).to("cuda:0")

refiner = StableDiffusionXLImg2ImgPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-refiner-1.0",
    torch_dtype     = torch.float16,
    use_safetensors = True,
    variant         = "fp16",
).to("cuda:1")

We can also use the [from_single_file()](https://huggingface.co/docs/diffusers/main/en/api/loaders/single_file#diffusers.loaders.FromSingleFileMixin.from_single_file) method to load a model checkpoint stored in a single file format (`.ckpt` or `.safetensors`) from the HF hub or locally.

```python
from   diffusers import StableDiffusionXLPipeline, StableDiffusionXLImg2ImgPipeline
import torch

pipeline = StableDiffusionXLPipeline.from_single_file(
  "https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/blob/main/sd_xl_base_1.0.safetensors",
  torch_dtype=torch.float16
).to("cuda")

refiner = StableDiffusionXLImg2ImgPipeline.from_single_file(
  "https://huggingface.co/stabilityai/stable-diffusion-xl-refiner-1.0/blob/main/sd_xl_refiner_1.0.safetensors",
  torch_dtype=torch.float16
).to("cuda")
```

## Text-to-image pipeline

For text-to-image, we condition the image generation on a text prompt. By default, SDXL generates a 1024x1024 image for the best results. We can try setting the `height` and `width` parameters to 768x768 or 512x512, but anything below 512x512 probably will not work.

In [ ]:
from PIL import Image

def image_grid(imgs, rows, cols):
    assert len(imgs) == rows * cols
    w, h           = imgs[0].size
    grid           = Image.new("RGB", size=(cols * w, rows * h))
    grid_w, grid_h = grid.size
    for i, img in enumerate(imgs):
        grid.paste(img, box=(i % cols * w, i // cols * h))
    return grid

In [ ]:
from   diffusers import StableDiffusionXLPipeline
import torch

prompt = "Astronaut in a jungle, cold color palette, muted colors, detailed, 8k"
image  = pipeline(prompt=prompt).images[0]

display(image)

## Image-to-image pipeline

For image-to-image, SDXL works especially well with image sizes between 768x768 and 1024x1024. In this case, we condition the image generation on the provided image and text prompt.

In [ ]:
from   diffusers       import StableDiffusionXLImg2ImgPipeline
from   diffusers.utils import load_image, make_image_grid
import torch

pipeline = StableDiffusionXLImg2ImgPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype     = torch.float16,
    variant         = "fp16",
    use_safetensors = True,
).to("cuda")

In [ ]:
seed        = 157
height      = 1024
width       = 1024
steps       = 50

url = [
    "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/sdxl-text2img.png",
    "https://raw.githubusercontent.com/CompVis/latent-diffusion/main/data/inpainting_examples/overture-creations-5sI6fQgYIuo.png"
]

prompt   = [
    "a dog catching a frisbee in the jungle",
    "A photography of a teenager on a bench, retro style, high resolution, 8k."
]

negative_prompt = [
    "five legs, single",
    "blurry, bad quality, not focused"
]

images      = []
init_images = []
for u, p, np in zip(url,prompt,negative_prompt):
    generator   = torch.Generator(device="cuda").manual_seed(seed)
    init_image  = load_image(u)

    image = pipeline(
        num_inference_steps = steps,
        height              = height,
        width               = width,
        prompt              = p,
        negative_prompt     = np,
        image               = init_image,
        strength            = 0.8,
        guidance_scale      = 10.5,
        generator           = generator,
    ).images[0]
    
    images.append(image)
    init_images.append(init_image)

In [ ]:
images_all = []
for inImg, img in zip(init_images,images):
    images_all.append(inImg)
    images_all.append(img)

image_grid(images_all, rows=2, cols=2)

## Inpainting pipeline

For inpainting, it is necessary to feed the model with an image, the mask of what we want to replace in the original image, and the prompt that describes what we want for filling the masked area.

In [ ]:
from   diffusers       import StableDiffusionXLInpaintPipeline
from   diffusers.utils import load_image, make_image_grid
import torch

with torch.no_grad():
    torch.cuda.empty_cache()

pipeline   = StableDiffusionXLInpaintPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype     = torch.float16,
    variant         = "fp16",
    use_safetensors = True,
).to("cuda")

In [ ]:
seed        = 157
height      = 1024
width       = 1024
steps       = 50
generator   = torch.Generator(device="cuda").manual_seed(seed)

#img_url    = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/sdxl-text2img.png"
#mask_url   = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/sdxl-inpaint-mask.png"

img_url     = "https://raw.githubusercontent.com/CompVis/latent-diffusion/main/data/inpainting_examples/overture-creations-5sI6fQgYIuo.png"
mask_url    = "https://raw.githubusercontent.com/CompVis/latent-diffusion/main/data/inpainting_examples/overture-creations-5sI6fQgYIuo_mask.png"
init_image  = load_image(img_url)
mask_image  = load_image(mask_url)
prompt      = "A majestic tiger sitting on a bench"

image  = pipeline(
    num_inference_steps = steps,
    height              = height,
    width               = width,
    prompt              = prompt,
    image               = init_image,
    mask_image          = mask_image,
    strength            = 0.85,
    guidance_scale      = 12.5,
    generator           = generator,
).images[0]

image512 = image.resize( [int(0.5 * s) for s in image.size] )
make_image_grid([init_image, mask_image, image512], rows=1, cols=3)

## Refine an image with the Refiner model

SDXL includes a [refiner model](https://huggingface.co/stabilityai/stable-diffusion-xl-refiner-1.0) specialized in denoising low-noise stage images to generate higher-quality images from the base model. There are two ways to use the refiner:

1. use the base and refiner models together to produce a refined image
2. use the base model to produce an image, and subsequently use the refiner model to add more details to the image (this is how SDXL was originally trained)

### Base model and refiner model

When we use the base model and refiner model together to generate an image, this is considered an [*ensemble of expert denoisers*](https://research.nvidia.com/labs/dir/eDiff-I/). The ensemble of expert denoisers approach requires fewer overall denoising steps than using the base model alone and later apply the image generated into the refiner model and, therefore, it should be significantly faster to run. However, you woould not be able to inspect the base model's output because it still contains a large amount of noise.

As an ensemble of expert denoisers, the base model serves as the expert during the high-noise diffusion stage and the refiner model serves as the expert during the low-noise diffusion stage.

We will now load the base and refiner models. Note that the refiner reuses the text encoder 2 and the VAE of the base model.

In [ ]:
from   diffusers import StableDiffusionXLPipeline, DiffusionPipeline
import torch

base = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype     = torch.float16,
    variant         = "fp16",
    use_safetensors = True
).to("cuda")

refiner = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-refiner-1.0",
    text_encoder_2  = base.text_encoder_2,
    vae             = base.vae,
    scheduler       = base.scheduler,
    torch_dtype     = torch.float16,
    use_safetensors = True,
    variant         = "fp16",
).to("cuda")

To use this approach, it is necessary to define the number of timesteps for each model to run through their respective stages. For the base model, this is controlled by the [`denoising_end`](https://huggingface.co/docs/diffusers/main/en/api/pipelines/stable_diffusion/stable_diffusion_xl#diffusers.StableDiffusionXLPipeline.__call__.denoising_end) argument and for the refiner model, it is controlled by the [`denoising_start`](https://huggingface.co/docs/diffusers/main/en/api/pipelines/stable_diffusion/stable_diffusion_xl#diffusers.StableDiffusionXLImg2ImgPipeline.__call__.denoising_start) argument.


> The `denoising_end` and `denoising_start` arguments should be a float between 0 and 1. These arguments represented the proportion of the discrete timesteps defined by the scheduler that will run on the base model and on the refiner. If we are also using the `strength` argument, it will be ignored because the number of denoising steps is determined by the discrete timesteps the model is trained on and the declared fractional cutoff.

Let us set `denoising_end=0.8` so the base model performs the initial 80% of denoising timesteps (those with high-level noise) and set `denoising_start=0.8` so the refiner model performs the remaining 20% of denoising timesteps (those with low-level noise). The base model output should be latents  instead of PIL images.

In [ ]:
# Inspect the architecture of the SDXL components
#
# base
# base.tokenizer
# base.tokenizer_2
# base.text_encoder
# base.text_encoder_2
# base.vae
# base.unet
#
# refiner.unet
#
# print(refiner.unet.config.time_embedding_type)
# print(base.text_encoder_2.config.projection_dim)

In [ ]:
from   diffusers.utils import load_image, make_image_grid

seed        = 157
height      = 1024
width       = 1024
steps       = 40
generator   = torch.Generator(device="cuda").manual_seed(seed)

#prompt = "A majestic lion jumping from a big stone at night"
prompt  = "Astronaut in a jungle, cold color palette, muted colors, detailed, 8k"

images = base(
    num_inference_steps = steps,
    height              = height,
    width               = width,
    prompt              = prompt,
    denoising_end       = 0.8,
    output_type         = "latent",
    generator           = generator,
).images

imageR = refiner(
    num_inference_steps = steps,
    height              = height,
    width               = width,
    prompt              = prompt,
    denoising_start     = 0.8,
    image               = images,
    generator           = generator,
).images[0]

In [ ]:
seed        = 157
height      = 1024
width       = 1024
steps       = 40
generator   = torch.Generator(device="cuda").manual_seed(seed)

#prompt = "A majestic lion jumping from a big stone at night"
prompt  = "Astronaut in a jungle, cold color palette, muted colors, detailed, 8k"

image = base(
    num_inference_steps = steps,
    height              = height,
    width               = width,
    prompt              = prompt,
    generator           = generator,
).images[0]

make_image_grid([image, imageR], rows=1, cols=2)

The ensemble of expert denoisers method works well for all available schedulers. 

SDXL gets a boost in image quality by using the refiner model to add additional high-quality details to the fully-denoised image from the base model, in an image-to-image setting.

We can use SDXL refiner with a different base model. For example, one can use the [Hunyuan-DiT](https://huggingface.co/docs/diffusers/main/en/using-diffusers/../api/pipelines/hunyuandit) or [PixArt-Sigma](https://huggingface.co/docs/diffusers/main/en/using-diffusers/../api/pipelines/pixart_sigma) pipelines to generate images with better prompt adherence. Once we have generated an image, it is passed to the SDXL refiner model to enhance its quality.


## Apply negative prompts

In [ ]:
from   diffusers import StableDiffusionXLPipeline, StableDiffusionXLImg2ImgPipeline
import torch

pipeline = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype     = torch.float16,
    variant         = "fp16",
    use_safetensors = True,
).to("cuda:0")

In [ ]:
seed        = 123
height      = 1024
width       = 1024
steps       = 50

prompt   = [
    "An astronaut riding a vibrant green horse, small pebbles in the ground, bright midaay sun, photo realistic",
    "An astronaut riding a vibrant green horse, small pebbles in the ground, bright midaay sun, photo realistic",
    "Hyper realistic photograph,photography of a child dressed like a gangster, black hat, 50 mm, film grain, Kodak portrait 800",
    "Hyper realistic photograph,photography of a child dressed like a gangster, black hat, 50 mm, film grain, Kodak portrait 800"
]
negative_prompt = [
    "",
    "five legs, multiple horses, moon",
    "",
    "jacket"
]

images      = []
for p, np in zip(prompt,negative_prompt):
    generator   = torch.Generator(device="cuda").manual_seed(seed)

    image = pipeline(
        num_inference_steps = steps,
        height              = height,
        width               = width,
        prompt              = p,
        negative_prompt     = np,
        #strength            = 0.8,
        #guidance_scale      = 10.5,
        generator           = generator,
    ).images[0]
    
    images.append(image)

In [ ]:
image_grid(images, rows=2, cols=2)

## Apply a different prompt to each text tokenizer/encoder

In [ ]:
from   diffusers import StableDiffusionXLPipeline, StableDiffusionXLImg2ImgPipeline
import torch

pipeline = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype     = torch.float16,
    variant         = "fp16",
    use_safetensors = True,
).to("cuda:0")

In [ ]:
seed        = 123
height      = 1024
width       = 1024
steps       = 50
prompt      = [
    "A child dressed like a gangster",
    "A child, gangster style."
]

prompt2     = [
    "Van Gogh painting.",
    "A black and white picture, film grain."
]
    
images      = []
for p, p2 in zip(prompt,prompt2):
    generator   = torch.Generator(device="cuda").manual_seed(seed)

    image = pipeline(
        num_inference_steps = steps,
        height              = height,
        width               = width,
        prompt              = p,  # prompt   is passed to OpenAI CLIP-ViT/L-14
        prompt_2            = p2, # prompt_2 is passed to OpenCLIP-ViT/bigG-14
        #guidance_scale      = 10.5,
        generator           = generator,
    ).images[0]    
    
    images.append(image)

In [ ]:
image_grid(images, rows=1, cols=2)

The dual text-encoders also support textual inversion embeddings that need to be loaded separately as explained in the [SDXL textual inversion](https://huggingface.co/docs/diffusers/main/en/using-diffusers/textual_inversion_inference#stable-diffusion-xl) section.

## Micro-conditioning

SDXL training involves several additional conditioning techniques, which are referred to as *micro-conditioning*. These include the original image size, the target image size, and cropping parameters. The micro-conditionings can be used at inference time to create high-quality, centered images.

We can use both micro-conditioning and negative prompt conditioning parameters due to classifier-free guidance. They are available in the [StableDiffusionXLPipeline](https://huggingface.co/docs/diffusers/main/en/api/pipelines/stable_diffusion/stable_diffusion_xl#diffusers.StableDiffusionXLPipeline), [StableDiffusionXLImg2ImgPipeline](https://huggingface.co/docs/diffusers/main/en/api/pipelines/stable_diffusion/stable_diffusion_xl#diffusers.StableDiffusionXLImg2ImgPipeline), [StableDiffusionXLInpaintPipeline](https://huggingface.co/docs/diffusers/main/en/api/pipelines/stable_diffusion/stable_diffusion_xl#diffusers.StableDiffusionXLInpaintPipeline), and [StableDiffusionXLControlNetPipeline](https://huggingface.co/docs/diffusers/main/en/api/pipelines/controlnet_sdxl#diffusers.StableDiffusionXLControlNetPipeline).

### Size conditioning

There are two types of size conditioning:

- [`original_size`](https://huggingface.co/docs/diffusers/main/en/api/pipelines/stable_diffusion/stable_diffusion_xl#diffusers.StableDiffusionXLPipeline.__call__.original_size) conditioning was included to allow training the model with different sized images, because it would be wasteful to discard the smaller images which make up almost 40% of the total training data. In this way, SDXL learns that upscaling artifacts are not supposed to be present in high-resolution images. During inference, we can use `original_size` to indicate the original image resolution. Using the default value of `(1024, 1024)` produces higher-quality images that resemble the 1024x1024 images from the training dataset. If we choose to use a lower resolution, such as `(256, 256)`, the model still generates 1024x1024 images, but they will look like the low resolution images (with simpler patterns and blurring) present in the training dataset.

- [`target_size`](https://huggingface.co/docs/diffusers/main/en/api/pipelines/stable_diffusion/stable_diffusion_xl#diffusers.StableDiffusionXLPipeline.__call__.target_size) conditioning aimed for SDXL to produce images of different aspect ratios. During inference, if we use the default value of `(1024, 1024)`, we will get an image that resembles the composition of square images in the dataset. We recommend using the same value for `target_size` and `original_size`, but feel free to experiment with other options.

🤗 Diffusers also allows us to specify a negative size conditioning guide generation away from certain image resolutions.

Next, we generate images conditioned on image resolutions of (128, 128), (256, 256), (512, 512), and (1024,1024).

In [ ]:
from   diffusers import StableDiffusionXLPipeline
import torch

pipeline = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype     = torch.float16,
    variant         = "fp16",
    use_safetensors = True,
).to("cuda")

In [ ]:
from torch import Generator

c_seed      = 123
prompt      = "Astronaut in a jungle, cold color palette, muted colors, detailed, 8k"
height      = [128, 256, 512, 1024]
width       = [128, 256, 512, 1024]

images = []
for i, (h,w) in enumerate(zip(height,width)):
    c_generator = torch.Generator(device="cuda").manual_seed(c_seed)
    image = pipeline(
        prompt        = prompt,
        original_size = (h, w),
        # negative_original_size = ( 512,  512),
        # negative_target_size   = (1024, 1024),
        generator     = c_generator,
    ).images[0]
    images.append(image)

In [ ]:
from IPython.display import display
from diffusers.utils import make_image_grid

grid = image_grid(images, rows=2, cols=2)
display(grid)

### Crop conditioning

Images generated by previous versions of Stable Diffusion sometimes appear to be cropped. This is because images are actually cropped during training so that all the images in a batch have the same size. By conditioning on crop coordinates, SDXL *learns* that no cropping, crop coordinates equal  `(0, 0)`, correlates with centered subjects and complete faces (this is the default value in 🤗 `diffusers`).

Thus, we can experiment with different crop coordinates for generating compositions that are not centered.

In [ ]:
from   diffusers import StableDiffusionXLPipeline
import torch

pipeline = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype     = torch.float16,
    variant         = "fp16",
    use_safetensors = True,
).to("cuda")

In [ ]:
seed            = 123

# prompt        = "Astronaut in a jungle, cold color palette, muted colors, detailed, 8k"
# prompt        = "hyper realistic photograph, full-body photography of a child dressed like a gangster, black hat, shiny shoes, 50 mm, film grain"
# prompt        = "An astronaut riding a pig."
prompt          = "A dog made of lego, in a grass field." 

#negative_prompt = "multiple children"
negative_prompt = "multiple, synthetic grass"

height      = 512
width       = 512
crop_h      = [0,     0, 256, 256]
crop_w      = [0,   256,   0, 256]
neg_crop_h  = [256, 256,   0,   0]
neg_crop_w  = [256,   0, 256,   0]
images = []
for i, (ch, cw, nch, ncw) in enumerate(zip(crop_h,crop_w, neg_crop_h, neg_crop_w)):
    print(f'cropH={ch} cropW={cw} negCropH={nch} negCropW={ncw}')
    c_generator = torch.Generator(device="cuda").manual_seed(seed)
    image = pipeline(
        height                         = height,
        width                          = width,
        prompt                         = prompt,
        negative_prompt                = negative_prompt,
        crops_coords_top_left          = (ch, cw),
        negative_crops_coords_top_left = (nch, ncw),
        guidance_scale                 = 8.0, # CFG guidance scale (default = 5.0, higher values mean higher text-image alignment)
        generator                      = c_generator,
    ).images[0]
    images.append(image)

In [ ]:
from diffusers.utils import make_image_grid
from IPython.display import display

grid = make_image_grid(images, rows=2, cols=2)
display(grid)

In [ ]:
# seed      = 123
# height    = 1024
# width     = 1024
#
# prompt     = "An astronaut in the desert, riding a light green horse, flat dunes, small stones on the ground, sunny day, detailed"
# prompt     = "An astronaut riding a vibrant green horse, small pebbles in the ground, bright midday sun, photo realistic."
# prompt     = "hyper realistic photograph, photography of a child dressed like a gangster, black hat, 50 mm, film grain, Kodak portrait 800"
# prompt     = "hyper realistic photograph, full-body photography of a children dressed like a gangster, black hat, shiny shoes, 50 mm, film grain"
# prompt     = "A close up portrait of an old village woman, white hair, olive green eyes, dramatic light, dreamlike light, photorealism, ultra detail, 85mm lens, indian village backdrop, long exposure background"
#
# negative_prompt = "five legs, multiple horses, moon"
# negative_prompt = "multiple"


In [ ]:
prompt = "Astronaut in a jungle, cold color palette, muted colors, detailed, 8k"

image  = pipeline(
    prompt                         = prompt,
    negative_original_size         = (512, 512),
    negative_crops_coords_top_left = (0, 0),
    negative_target_size           = (1024, 1024),
).images[0]

image

SDXL uses two text-encoders, so it is possible to pass a different prompt to each text-encoder, which can [improve quality](https://github.com/huggingface/diffusers/issues/4004#issuecomment-1627764201). So, we pass our original prompt as `prompt` and the second prompt as `prompt_2`. We can use `negative_prompt` and `negative_prompt_2` to specify negative prompts.

In [ ]:
from   diffusers import StableDiffusionXLPipeline
import torch

pipeline = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype     = torch.float16,
    variant         = "fp16",
    use_safetensors = True,
).to("cuda")

## Using different aspect ratios

SDXL was trained to handle diverse aspect ratios. Instead of generating only 1:1 images, it can generate images with popular aspect ratios such as 16:9. So, let us generate an image in roughly that aspect ratio, note that it is not exact, as we have to abide by the rule that the height and width are divisible by 8.

In [ ]:
from   diffusers import StableDiffusionXLPipeline, DiffusionPipeline
import torch

pipeline = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype     = torch.float16,
    variant         = "fp16",
    use_safetensors = True
).to("cuda")

In [ ]:
seed            = 123
steps           = 50
width           = 1344
height          = 768

positive_prompt = "Beautiful mountain landscape, dawn, victorian painting, high quality"
negative_prompt = "Bad quality, artifacts, low quality, lowres, deformed, malformed, photograph, hyperrealism, photo, bad artist"

generator       = torch.Generator(device="cuda").manual_seed(seed)

image = pipeline(
    width               = width,
    height              = height,
    num_inference_steps = steps,
    prompt              = positive_prompt,
    negative_prompt     = negative_prompt,
    generator           = generator
).images[0]

display(image)

In [ ]:
seed            = 123
steps           = 50
width           = 768
height          = 1344

positive_prompt = [
    "City skyscrapers, Hong Kong, 8k, high quality",
    "City street photography, tall buildings, trees, fall season, 8k, high quality"
]
negative_prompt = [
    "",
    "night, green leaves"
]

images = []
for i, (pp,np) in enumerate(zip(positive_prompt,negative_prompt)):
    generator = torch.Generator(device="cuda").manual_seed(seed)
    image       = pipeline(
        width               = width,
        height              = height,
        num_inference_steps = steps,
        prompt              = pp,
        negative_prompt     = np,
        generator           = generator
    ).images[0]
    images.append(image)

In [ ]:
from diffusers.utils import make_image_grid
from IPython.display import display

grid = make_image_grid(images, rows=1, cols=2)
display(grid)

## Change the noise scheduler

By default, SDXL uses the `EulerDiscreteScheduler`. However, other schedulers may be interesting, as some produce a higher quality, allow faster inference, or need less steps for denoising a latent. So, let us see how we can do that. First, we will check out the compatible schedulers.

In [ ]:
from   diffusers import StableDiffusionXLPipeline, DiffusionPipeline
import torch

pipeline = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype     = torch.float16,
    variant         = "fp16",
    use_safetensors = True
).to("cuda")

In [ ]:
print(f'Current base scheduler:    {pipeline.scheduler}')

lst_compat = for sched in lst_compat:
    print(sched)
type(pipeline.scheduler).__name__

When this notebook was written, these were the compatible schedulers:

```
[
 diffusers.schedulers.scheduling_unipc_multistep.UniPCMultistepScheduler,
 diffusers.schedulers.scheduling_edm_euler.EDMEulerScheduler,
 diffusers.schedulers.scheduling_dpmsolver_singlestep.DPMSolverSinglestepScheduler,
 diffusers.schedulers.scheduling_ddim.DDIMScheduler,
 diffusers.utils.dummy_torch_and_torchsde_objects.DPMSolverSDEScheduler,
 diffusers.schedulers.scheduling_k_dpm_2_ancestral_discrete.KDPM2AncestralDiscreteScheduler,
 diffusers.schedulers.scheduling_heun_discrete.HeunDiscreteScheduler,
 diffusers.schedulers.scheduling_euler_ancestral_discrete.EulerAncestralDiscreteScheduler,
 diffusers.schedulers.scheduling_dpmsolver_multistep.DPMSolverMultistepScheduler,
 diffusers.schedulers.scheduling_ddpm.DDPMScheduler,
 diffusers.schedulers.scheduling_k_dpm_2_discrete.KDPM2DiscreteScheduler,
 diffusers.schedulers.scheduling_lms_discrete.LMSDiscreteScheduler,
 diffusers.schedulers.scheduling_deis_multistep.DEISMultistepScheduler,
 diffusers.schedulers.scheduling_pndm.PNDMScheduler,
 diffusers.schedulers.scheduling_euler_discrete.EulerDiscreteScheduler
]
```

Next, we compare nine of these schedulers in terms of computation time and generated image.

We simply set the scheduler by changing the property `scheduler` of the pipeline to the new scheduler and pass to it our configuration of the `EulerDiscreteScheduler`. We can also change the configuration.


In [ ]:
from time      import time
from diffusers import (
    UniPCMultistepScheduler,
    EulerDiscreteScheduler,
    DDIMScheduler,
    KDPM2AncestralDiscreteScheduler,
    HeunDiscreteScheduler,
    DDPMScheduler,
    KDPM2DiscreteScheduler,
    LMSDiscreteScheduler,
    PNDMScheduler
)

seed            = 123
steps           = 50
width           = 1024
height          = 1024
positive_prompt = "A majestic lion jumping from a big stone at night"
negative_prompt = "realistic, realism, photograph"

scheds = [
    UniPCMultistepScheduler,
    EulerDiscreteScheduler,
    DDIMScheduler,
    KDPM2AncestralDiscreteScheduler,
    HeunDiscreteScheduler,
    DDPMScheduler,
    KDPM2DiscreteScheduler,
    LMSDiscreteScheduler,
    PNDMScheduler
]

images = []
for sched in scheds:
    pipeline.scheduler = sched.from_config(
        pipeline.scheduler.config
    )
    print(f'Current base scheduler:    {sched.__name__}')
    generator       = torch.Generator(device="cuda").manual_seed(seed)
    startT = time()
    
    image = pipeline(
        width               = width,
        heigh               = height,
        num_inference_steps = steps,
        prompt              = positive_prompt,
        negative_prompt     = negative_prompt,
        generator           = generator
    ).images[0]
    endT = time()
    images.append(image)
    print(f'Computation time ({sched.__name__}): {endT - startT :.1f} s')

In [ ]:
from diffusers.utils import make_image_grid
from IPython.display import display

grid = make_image_grid(images, rows=3, cols=3)
display(grid)

You can inspect the values of the times that correspond to the schedule we just defined, relative to the 1000 steps used during training.

In [ ]:
pipeline.scheduler = EulerDiscreteScheduler.from_config(
    pipeline.scheduler.config
)

In [ ]:
# Print the 50 times of our sampling schedule relative
# to the original 1000 steps used during training of the U-Net
print(pipeline.scheduler.timesteps)

We can also inspect the amount of noise (variance) associated to each sampling time.

In [ ]:
# Print the noise level associated with the 50 sampling times
print(pipeline.scheduler.sigmas)

During sampling, we start at a high noise level (in fact, our input latent is pure noise) and gradually denoise the noisy latent into a "clean" latent, according to this noise variance schedule.

In [ ]:
import matplotlib.pyplot as plt

# Plot the 50-values noise schedule
plt.plot(pipeline.scheduler.sigmas)
plt.title('Noise schedule')
plt.xlabel('sampling step')
plt.ylabel('noise variance')
plt.show()

In [ ]:
# Plot the 50 times of the noise schedule
plt.plot(pipeline.scheduler.timesteps)
plt.title('Noise schedule')
plt.xlabel('sampling step')
plt.ylabel('denoising timestep')
plt.show()

In [ ]:
pipeline.scheduler = DDIMScheduler.from_config(pipeline.scheduler.config)
print(f'Current base scheduler:    {pipeline.scheduler._class_name}')

pipeline.scheduler.set_timesteps(50)
print(f'Timesteps: {pipeline.scheduler.timesteps}')

pipeline.scheduler.sigmas = []
for ts in pipeline.scheduler.timesteps:
    prev_ts  = ts - pipeline.scheduler.num_train_timesteps // pipeline.scheduler.num_inference_steps
    variance = pipeline.scheduler._get_variance(ts, prev_ts)
    pipeline.scheduler.sigmas.append(variance.item())
    print(f'{variance :.4f}', end='  ')

plt.plot(pipeline.scheduler.sigmas)
plt.title('DDIM scheduler')
plt.xlabel('sampling step')
plt.ylabel('noise variance')
plt.show()

In [ ]:
pipeline.scheduler = DDPMScheduler.from_config(pipeline.scheduler.config)
print(f'Current base scheduler:    {pipeline.scheduler._class_name}')

pipeline.scheduler.set_timesteps(50)
print(f'Timesteps: {pipeline.scheduler.timesteps}')

pipeline.scheduler.sigmas = []
for ts in pipeline.scheduler.timesteps:
    variance = pipeline.scheduler._get_variance(ts)
    pipeline.scheduler.sigmas.append(variance.item())
    print(f'{variance :.4f}', end='  ')

plt.plot(variance)
plt.title('DDPM scheduler')
plt.xlabel('sampling step')
plt.ylabel('noise variance')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

scheds = [
    UniPCMultistepScheduler,
    EulerDiscreteScheduler,
    DDIMScheduler,
    KDPM2AncestralDiscreteScheduler,
    HeunDiscreteScheduler,
    DDPMScheduler,
    KDPM2DiscreteScheduler,
    LMSDiscreteScheduler,
    PNDMScheduler
]

for sched in scheds:
    pipeline.scheduler = sched.from_config(
        pipeline.scheduler.config
    )
    sched_name = pipeline.scheduler._class_name
    print(f'Current base scheduler:    {sched_name}')

    # Setting the number of sampling steps
    pipeline.scheduler.set_timesteps(50)

    # Print the 50 times of our sampling schedule relative
    # to the original 1000 steps used during training of the U-Net
    # print(f'Timesteps: {pipeline.scheduler.timesteps}')

    if sched_name == 'DDPMScheduler': # continuous scheduler that does not have sigmas
        pipeline.scheduler.sigmas = []
        for ts in pipeline.scheduler.timesteps:
            variance = pipeline.scheduler._get_variance(ts)
            pipeline.scheduler.sigmas.append(variance.item())

    elif sched_name == 'DDIMScheduler':  # continuous scheduler that does not have sigmas
        pipeline.scheduler.sigmas = []
        for ts in pipeline.scheduler.timesteps:
            prev_ts  = ts - pipeline.scheduler.num_train_timesteps // pipeline.scheduler.num_inference_steps
            variance = pipeline.scheduler._get_variance(ts, prev_ts)
            pipeline.scheduler.sigmas.append(variance.item())
    
    # --- style configuration ---
    plt.rcParams.update({
        "figure.figsize": (10, 4),   # Width x Height (inches)
        "axes.grid": True,
    })
   
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True, sharex=False)
    
    # --- noise variance (sigmas) ---
    ax0 = axes[0]
    if hasattr(pipeline.scheduler,'sigmas'):
        ax0.plot(pipeline.scheduler.sigmas, marker='o', linewidth=1)
    ax0.set_title(f'{sched_name} — sigma')
    ax0.set_xlabel('sampling step')
    ax0.set_ylabel('noise variance')
    
    # ---------- betas -----------
    ax1 = axes[1]
    ax1.plot(pipeline.scheduler.betas, marker='o', color='tab:orange', linewidth=1)
    ax1.set_title(f'{sched_name} — beta')
    ax1.set_xlabel('sampling step')
    ax1.set_ylabel('beta')
   
    # save to file
    fname = f'sdxl_sigma_beta_{sched_name}.png'
    fig.savefig(fname, dpi=200, bbox_inches="tight")
    plt.show()


## Using extended token lengths and token weights in prompts with Compel

In Stable Diffusion 1.x, 2,x and SDXL the length of the prompts is limited to 77 tokens. This is due a limitation in the text embedding tensors. However, we can use a library such as `compel` to extend the token sequence and also associate weights to the tokens.

First, we create a prompt embedder/weighter with Compel using the tokenizer and text encoder of the base model of SDXL. We then specify the returned embeddings type to be `ReturnedEmbeddingsType.PENULTIMATE_HIDDEN_STATES_NON_NORMALIZED` and set `requires_pooled` equal to `False`. Next, we create a second prompt embedder/weighter with Compel using the tokenizer and text encoder of the Refiner Model. We then specify the returned embeddings type to be `ReturnedEmbeddingsType.PENULTIMATE_HIDDEN_STATES_NON_NORMALIZED` and set `requires_pooled` equal to `True`. Then, we can create the normal and pooled prompt embeddings. We can then pass them to the model through the arguments `prompt_embeds` and `pooled_prompt_embeds`.

Simply, we can add `+n` or `-n` at the end of a word, to specify weights equal to 1.1^n (`+`) or 0.9^n (`-`) where n is equal to the amount following `+` or `-`. If we do not put multiple words in brackets `()` we apply the weighting to the word that represents the token embeddings associated to it. We can also specify the weight directly, for example, `(victorian)1.2`. If we do not want funky results, it is probably a good idea to stay in the range of 0.6:1.5.

Compel also offers other tricks, that may be interesting, take a look [here](https://github.com/damian0815/compel/blob/main/doc/syntax.md).

Let us ensure that we have `compel` installed and use compatible versions of `transformers`and `huggingface-hub`.

In [ ]:
!pip install --upgrade transformers==4.57.6
!pip install --upgrade diffusers
!pip install --upgrade huggingface-hub==0.36.2
!pip install --upgrade jedi==0.19.2 
!pip install --upgrade compel==2.3.1

In [ ]:
from   diffusers import StableDiffusionXLPipeline, DiffusionPipeline
import torch

pipeline = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0", 
    variant         = "fp16", 
    use_safetensors = True,
    torch_dtype     = torch.float16
).to("cuda")

### First example

Using a long prompt but without Compel.

In [ ]:
seed       = 123
steps      = 50
height     = 768
width      = 768
prompt     = "photo, awesome, enchantress in her cluttered workshop, grey background, headshot, head. tabletop roleplaying artwork. Dynamic, high-fantasy digital painting with dramatic lighting, painterly textures, and intricate detail. intense action, and blends realism with impressionistic brushstrokes for an immersive composition. concept art, clean outlines, painterly, highlights, shadows, blue ray of light enters through the window,"
generator  = torch.Generator(device="cuda").manual_seed(seed)

image = pipeline(
    width               = width,
    height              = height,
    num_inference_steps = steps,
    prompt              = prompt,
    #negative_prompt     = negative_prompt,
    generator           = generator
).images[0]

display(image)

Using Compel, and try three different prompts to play with weights.

In [ ]:
from compel import CompelForSDXL

seed       = 123
steps      = 50
height     = 768
width      = 768
generator  = torch.Generator(device="cuda").manual_seed(seed)

prompt  = "photo, awesome, enchantress in her cluttered workshop, grey background, headshot, head. tabletop roleplaying artwork. Dynamic, high-fantasy digital painting with dramatic lighting, painterly textures, and intricate detail. intense action, and blends realism with impressionistic brushstrokes for an immersive composition. concept art, clean outlines, painterly, highlights, shadows, blue ray of light enters through the window,"
#prompt = "photo, awesome, enchantress in her cluttered workshop, grey background, headshot, head+. tabletop roleplaying artwork. Dynamic, high-fantasy digital painting with dramatic lighting, painterly textures, and intricate detail. intense action, and blends realism with impressionistic brushstrokes for an immersive composition. concept art, clean outlines, painterly, highlights, shadows, (blue ray of light)+ enters through the window. (detailed hands)+"
#prompt = "photo, awesome, enchantress+ in her cluttered workshop, grey background, headshot, head++. tabletop roleplaying artwork. Dynamic, high-fantasy digital painting with dramatic lighting, painterly textures, and intricate detail. intense action, and blends realism with impressionistic brushstrokes for an immersive composition. concept art, clean outlines, painterly, highlights, shadows, (blue ray of light)++ enters through the window. (detailed hands)++"

compel                 = CompelForSDXL(pipeline)
conditioning           = compel(prompt)

#negative_conditioning = compel(negative_prompt)
#pipeline.enable_sequential_cpu_offload()

# We can also use a batched input:
# conditioning = compel(["A cat playing with a ball in the forest", "deformed, ugly"])
# and then use conditioning.embeds[0:1] for positive and conditioning.embeds[1:2] for negative

image = pipeline (
    width                         = width,
    height                        = height,
    num_inference_steps           = steps,
    prompt_embeds                 = conditioning.embeds,
    pooled_prompt_embeds          = conditioning.pooled_embeds,
    #negative_prompt_embeds        = negative_conditioning.embeds, 
    #negative_pooled_prompt_embeds = negative_conditioning.pooled_embeds,
).images[0]

display(image)

### Second example

Using a long prompt but without Compel.

In [ ]:
from IPython.display import display

seed            = 123
steps           = 50
width           = 768   # Reduced width
height          = 768   # Reduced height
positive_prompt = "Anime Portrait of a young woman, pretty stunning blue eyes, soft face, long pink hair, bow hairclip, pink school uniform, high quality, hq, 4k, 8k, award-winning, happy white cat in her arms, dawn, close-up, cherry blossoms, wearing golden blossom earrings, bokeh effect, cute, lake with white cranes in the background, butterflies, reflections"
negative_prompt = "Bad quality, artifacts, low quality, lowres, deformed, malformed, extra hands, extra limbs, bad eyes, ugly, lowres, washed out, poor quality"
generator       = torch.Generator(device="cuda").manual_seed(seed)

image = pipeline(
    width               = width,
    height              = height,
    num_inference_steps = steps,
    prompt              = positive_prompt,
    negative_prompt     = negative_prompt,
    generator           = generator
).images[0]

display(image)

Using Compel and a first version of the long prompt.

In [ ]:
from compel import CompelForSDXL

seed            = 123
steps           = 50
height          = 768   # Reduced height
width           = 768   # Reduced width
positive_prompt = "(Anime Portrait)1.2 of a young woman, pretty+ stunning+ blue eyes, soft face, long pink hair, bow hairclip, pink school uniform, high quality, hq, 4k, 8k, award-winning, happy white cat in her arms, dawn+, close-up+, (cherry blossoms)---, (wearing golden blossom earrings)+, (bokeh effect)-, cute, lake with (white cranes)++ in the background, butterflies--, reflections--"
negative_prompt = "Bad quality, artifacts, low quality, lowres, deformed+, malformed+, extra hands, extra limbs, bad eyes, ugly, lowres, washed out, poor quality"

generator       = torch.Generator(device="cuda").manual_seed(seed)

compel = CompelForSDXL(pipeline)

# use batched input - Compel will automatically pad the shorter main prompt to
# the length of the longer negative prompt (or vice versa), otherwise we will
# have to use `pad_conditioning_tensors_to_same_length` from `compel.utils`
conditioning = compel([positive_prompt, negative_prompt])

print(conditioning.embeds.shape, conditioning.pooled_embeds.shape)

image = pipeline(
  prompt_embeds                 = conditioning.embeds[0:1],
  pooled_prompt_embeds          = conditioning.pooled_embeds[0:1],
  negative_prompt_embeds        = conditioning.embeds[1:2],
  negative_pooled_prompt_embeds = conditioning.pooled_embeds[1:2],
  num_inference_steps           = steps,
  width                         = height,
  height                        = width,
  generator                     = generator,
).images[0]

display(image)

Using Compel and a second version of the long prompt.

In [ ]:
from compel import CompelForSDXL

seed            = 123
steps           = 50
height          = 768   # Reduced height
width           = 768   # Reduced width
positive_prompt = "(Anime Portrait)1.2 of a young woman, pretty+ stunning+ blue eyes, soft face, long pink hair, bow hairclip, pink school uniform, high quality, happy white cat in her arms"
negative_prompt = "Bad quality, artifacts, low quality, lowres, deformed+, malformed+, extra hands, extra limbs, bad eyes, ugly, lowres, washed out, poor quality"

generator       = torch.Generator(device="cuda").manual_seed(seed)

compel          = CompelForSDXL(pipeline)

# use batched input - Compel will automatically pad the shorter main prompt to
# the length of the longer negative prompt (or vice versa), otherwise we will
# have to use `pad_conditioning_tensors_to_same_length` from `compel.utils`
conditioning = compel([positive_prompt, negative_prompt])

print(conditioning.embeds.shape, conditioning.pooled_embeds.shape)

image = pipeline(
  prompt_embeds                 = conditioning.embeds[0:1],
  pooled_prompt_embeds          = conditioning.pooled_embeds[0:1],
  negative_prompt_embeds        = conditioning.embeds[1:2],
  negative_pooled_prompt_embeds = conditioning.pooled_embeds[1:2],
  num_inference_steps           = steps,
  width                         = height,
  height                        = width,
  generator                     = generator,
).images[0]

display(image)

Using Compel and the Refiner model.

As mensioned previously, SDXL introduced an optional Refiner Model that adds some details to improve the perceived quality of the image. This step requires significantly higher amounts of VRAM, so it may be necessary to activate CPU offloading. Let us proceed and define the refiner pipeline.

In [ ]:
# Refiner
refiner_pipeline = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-refiner-1.0",
    text_encoder_2  = pipeline.text_encoder_2,
    vae             = pipeline.vae,
    torch_dtype     = torch.float16,
    scheduler       = pipeline.scheduler,
    use_safetensors = True,
    variant         = "fp16",
).to("cuda")

#refiner_pipeline.enable_model_cpu_offload()

If we have a large VRAM, we can load the refiner_pipeline into the GPU instead of activating cpu offloading like using `refiner_pipe.to("cuda", torch.float16)`. To save some GPU memory, we utilize the second text encoder, the vae and the scheduler from the base model.

In [ ]:
seed            = 123
steps           = 50
width           = 1024
height          = 1024
positive_prompt = "(Anime Portrait)1.2 of a young woman, pretty+ stunning+ blue eyes, soft face, long pink hair, bow hairclip, pink school uniform, high quality, happy white cat in her arms"
negative_prompt = "Bad quality, artifacts, low quality, lowres, deformed+, malformed+, extra hands, extra limbs, bad eyes, ugly, lowres, washed out, poor quality"
denoise_frac    = 0.8 # 80% of denoising is done on base pipeline, the remaining 20% on refiner

generator = torch.Generator(device="cuda").manual_seed(seed)

compel          = CompelForSDXL(pipeline)

# use batched input - Compel will automatically pad the shorter main prompt to
# the length of the longer negative prompt (or vice versa), otherwise we will
# have to use `pad_conditioning_tensors_to_same_length` from `compel.utils`
conditioning = compel([positive_prompt, negative_prompt])

print(conditioning.embeds.shape, conditioning.pooled_embeds.shape)

latent = pipeline(
    width                         = width,
    height                        = height,
    prompt_embeds                 = conditioning.embeds[0:1],
    pooled_prompt_embeds          = conditioning.pooled_embeds[0:1],
    negative_prompt_embeds        = conditioning.embeds[1:2],
    negative_pooled_prompt_embeds = conditioning.pooled_embeds[1:2],
    generator                     = generator,
    output_type                   = "latent",
    denoising_end                 = denoise_frac
).images

image = refiner_pipeline(
    width                         = width,
    height                        = height,
    prompt                        = positive_prompt,
    negative_prompt               = negative_prompt,
    num_inference_steps           = steps,
    image                         = latent,
    generator                     = generator,
    denoising_start               = denoise_frac
).images[0]

display(image)

### Third example

Let us focus on modifying token's weights only.

In [ ]:
from compel import CompelForSDXL

seed   = 123
steps  = 50
height = 512
width  = 512

prompt = [
    "a color drawing of man with a ragsack, (mountains)0.6 in the background",
    "a color drawing of man with a ragsack, (mountains)0.7 in the background",
    "a color drawing of man with a ragsack, (mountains)0.8 in the background",
    "a color drawing of man with a ragsack, (mountains)0.9 in the background",
    "a color drawing of man with a ragsack, mountains in the background",
    "a color drawing of man with a ragsack, (mountains)1.1 in the background"
]

compel = CompelForSDXL(pipeline)

images = []
for p in prompt:
    generator     = torch.Generator(device="cuda").manual_seed(seed)
    conditioning  = compel(p)

    image = pipeline (
        width                         = width,
        height                        = height,
        num_inference_steps           = steps,
        prompt_embeds                 = conditioning.embeds,
        pooled_prompt_embeds          = conditioning.pooled_embeds,
        generator                     = generator
    ).images[0]
    images.append(image)

In [ ]:
from diffusers.utils import make_image_grid

make_image_grid(images, rows=3, cols=2)

## Optimizations

SDXL is a large model, and we may need to optimize memory to get it to run on our hardware. Here are some tips to save memory and speed up inference.

1. Offload the model to the CPU with [enable_model_cpu_offload()](https://huggingface.co/docs/diffusers/main/en/api/pipelines/overview#diffusers.DiffusionPipeline.enable_model_cpu_offload) for out-of-memory errors:

```diff
- base.to("cuda")
- refiner.to("cuda")
+ base.enable_model_cpu_offload()
+ refiner.enable_model_cpu_offload()
```

2. Use `torch.compile` for approximatelly a 20% speed-up (requires `torch>=2.0`):

```diff
+ base.unet = torch.compile(base.unet, mode="reduce-overhead", fullgraph=True)
+ refiner.unet = torch.compile(refiner.unet, mode="reduce-overhead", fullgraph=True)
```

3. Enable [xFormers](https://huggingface.co/docs/diffusers/main/en/using-diffusers/../optimization/xformers) to run SDXL if `torch<2.0`:

```diff
+ base.enable_xformers_memory_efficient_attention()
+ refiner.enable_xformers_memory_efficient_attention()
```

## Other resources

For experimenting with a minimal version of the [UNet2DConditionModel](https://huggingface.co/docs/diffusers/main/en/api/models/unet2d-cond#diffusers.UNet2DConditionModel) used in SDXL, take a look at the [minSDXL](https://github.com/cloneofsimo/minSDXL) implementation which is written in PyTorch and directly compatible with 🤗 `diffusers`.